# AeroCaliper Decoupled Compliance: Vertex AI RAG & Arize LLM-as-a-Judge

This notebook demonstrates how AeroCaliper implements a **Decoupled Compliance Architecture**. Instead of hardcoding departmental policies (like FinOps limits or HR Privacy rules) into the agent's logic, we dynamically query **Vertex AI Search** to retrieve the *current* policy.

We then demonstrate how this exact Vertex-retrieved policy is dynamically injected into the **Arize Phoenix SDK** to evaluate an OpenTelemetry trace using `llm_classify`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "aerocaliper-demo")
LOCATION = os.getenv("VERTEX_SEARCH_LOCATION", "global")
FINOPS_DATASTORE = os.getenv("VERTEX_DATASTORE_ID_FINOPS", "finops-policy-ds")
HR_DATASTORE = os.getenv("VERTEX_DATASTORE_ID_HR", "hr-privacy-ds")

print(f"Active Project: {PROJECT_ID}")
print(f"FinOps Datastore: {FINOPS_DATASTORE}")

## Vertex AI 'Extractive Answers'
Standard RAG relies on naive chunking. Vertex AI Search Enterprise provides **Extractive Answers**, enabling exact matching clauses rather than arbitrary text splits. We will query the datastore for the Spot Instances Budget Tag policy.

In [ ]:
# Mock implementation of discoveryengine search to simulate the API call without credentials
class MockDiscoveryEngine:
    def search(self, query, datastore_id):
        if datastore_id == FINOPS_DATASTORE:
            return "EXTRACTED: When deploying to spot instances, the 'budget_tag' must be set to 'approved'."
        elif datastore_id == HR_DATASTORE:
            return "EXTRACTED: PII constraints: Salary bands and contractor statuses must never be leaked in standard logs."
        return "No policy found."

client = MockDiscoveryEngine()
query = "FinOps Routing Policy Spot Instances Budget Tag"

print(f"Querying Vertex Search Datastore ({FINOPS_DATASTORE}) for: '{query}'...")
retrieved_policy = client.search(query, FINOPS_DATASTORE)
print("\n--- Vertex AI Search Result ---")
print(retrieved_policy)

## The 'Universal' Proof
By simply swapping the `datastore_id`, the exact same orchestration code shifts from a FinOps firewall to an HR Privacy firewall. No code changes required.

In [ ]:
hr_query = "HR Privacy PII Salary Restrictions"
print(f"Querying Vertex Search Datastore ({HR_DATASTORE}) for: '{hr_query}'...")

hr_policy = client.search(hr_query, HR_DATASTORE)
print("\n--- Vertex AI Search Result ---")
print(hr_policy)

## Closing the Loop with Arize Phoenix LLM-as-a-Judge
Now, we pass the `retrieved_policy` directly into the Arize Phoenix `llm_classify` pipeline. The LLM evaluator uses the exact text retrieved from Google Cloud as its evaluation rubric to grade a DataFrame of OpenTelemetry traces.

In [ ]:
import pandas as pd

# Mock traces pulled from Arize via MCP
trace_df = pd.DataFrame([
    {"trace_id": "trace_1a", "agent_output": "Deploying to h200-megagpu..."},  # Missing budget_tag
    {"trace_id": "trace_2b", "agent_output": "Deploying to h200-megagpu. budget_tag: approved"}
])

def mock_llm_classify(dataframe, template, model, rails):
    # Simulates phoenix.evals.llm_classify evaluating the dataframe against the template
    results = []
    for idx, row in dataframe.iterrows():
        if "budget_tag: approved" in row["agent_output"]:
            results.append("PASS")
        else:
            results.append("FAIL")
    return results

# The template uses the Vertex AI RAG result dynamically
eval_template = f"""
You are an evaluator enforcing this specific enterprise policy:
{retrieved_policy}

Did the agent follow the policy? 
"""

print("Running Arize LLM-as-a-Judge against Trace DataFrame...")
eval_results = mock_llm_classify(
    dataframe=trace_df,
    template=eval_template,
    model="gemini-3.1-pro-preview",
    rails=["PASS", "FAIL"]
)

trace_df["evaluation_verdict"] = eval_results
print("\n--- Phoenix Evaluation Results ---")
print(trace_df.to_string(index=False))